# Module 7 Homework - Streaming with Kafka and PyFlink

Green Taxi Trip data from October 2025.

**Setup**: Run the workshop docker-compose first:
```bash
cd 07-streaming/workshop/
docker compose build
docker compose up -d
```

## Question 1: Redpanda Version

```bash
docker exec -it workshop-redpanda-1 rpk version
```

**Answer: v25.3.9**

## Question 2: Sending data to Redpanda

Create the topic:
```bash
docker exec -it workshop-redpanda-1 rpk topic create green-trips
```

In [ ]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = [
    'lpep_pickup_datetime', 'lpep_dropoff_datetime',
    'PULocationID', 'DOLocationID', 'passenger_count',
    'trip_distance', 'tip_amount', 'total_amount'
]
df = pd.read_parquet(url, columns=columns)
print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
import json
from time import time
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=json_serializer
)

t0 = time()

for _, row in df.iterrows():
    message = row.to_dict()
    # Convert datetime columns to strings for JSON serialization
    message['lpep_pickup_datetime'] = str(message['lpep_pickup_datetime'])
    message['lpep_dropoff_datetime'] = str(message['lpep_dropoff_datetime'])
    producer.send('green-trips', value=message)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

**Answer: ~60 seconds**

## Question 3: Consumer - trip distance

Count trips with `trip_distance` > 5.0 km.

In [ ]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'green-trips',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest',
    group_id='green-trips-counter',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=10000
)

count = 0
total = 0
for message in consumer:
    trip = message.value
    total += 1
    if float(trip['trip_distance']) > 5.0:
        count += 1

consumer.close()
print(f"Total trips: {total}")
print(f"Trips with distance > 5 km: {count}")

**Answer: 8506**

## Part 2: PyFlink (Questions 4-6)

For the PyFlink questions, create job files in `workshop/src/job/` and submit them to Flink.

Important notes:
- The `green-trips` topic has 1 partition, so set `env.set_parallelism(1)`
- Convert datetime strings: `TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss')`
- Submit jobs: `docker exec -it workshop-jobmanager-1 flink run -py /opt/src/job/your_job.py`
- Let jobs run for 1-2 minutes, then query results in PostgreSQL

### Create PostgreSQL tables

```sql
-- Q4: Tumbling window by pickup location
CREATE TABLE tumbling_pickup (
    window_start TIMESTAMP,
    PULocationID INTEGER,
    num_trips BIGINT,
    PRIMARY KEY (window_start, PULocationID)
);

-- Q5: Session window by pickup location
CREATE TABLE session_pickup (
    session_start TIMESTAMP,
    session_end TIMESTAMP,
    PULocationID INTEGER,
    num_trips BIGINT
);

-- Q6: Tumbling window tip amount
CREATE TABLE hourly_tips (
    window_start TIMESTAMP,
    total_tips DOUBLE PRECISION,
    PRIMARY KEY (window_start)
);
```

## Question 4: Tumbling window - pickup location

5-minute tumbling window counting trips per `PULocationID`.

Save as `src/job/q4_tumbling_pickup.py`:

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import EnvironmentSettings, StreamTableEnvironment


def create_source(t_env):
    t_env.execute_sql("""
        CREATE TABLE green_trips (
            lpep_pickup_datetime VARCHAR,
            lpep_dropoff_datetime VARCHAR,
            PULocationID INTEGER,
            DOLocationID INTEGER,
            passenger_count DOUBLE,
            trip_distance DOUBLE,
            tip_amount DOUBLE,
            total_amount DOUBLE,
            event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
            WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
        ) WITH (
            'connector' = 'kafka',
            'properties.bootstrap.servers' = 'redpanda:29092',
            'topic' = 'green-trips',
            'scan.startup.mode' = 'earliest-offset',
            'properties.auto.offset.reset' = 'earliest',
            'format' = 'json'
        );
    """)


def create_sink(t_env):
    t_env.execute_sql("""
        CREATE TABLE tumbling_pickup (
            window_start TIMESTAMP(3),
            PULocationID INT,
            num_trips BIGINT,
            PRIMARY KEY (window_start, PULocationID) NOT ENFORCED
        ) WITH (
            'connector' = 'jdbc',
            'url' = 'jdbc:postgresql://postgres:5432/postgres',
            'table-name' = 'tumbling_pickup',
            'username' = 'postgres',
            'password' = 'postgres',
            'driver' = 'org.postgresql.Driver'
        );
    """)


def run():
    env = StreamExecutionEnvironment.get_execution_environment()
    env.enable_checkpointing(10 * 1000)
    env.set_parallelism(1)

    settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
    t_env = StreamTableEnvironment.create(env, environment_settings=settings)

    create_source(t_env)
    create_sink(t_env)

    t_env.execute_sql("""
        INSERT INTO tumbling_pickup
        SELECT
            window_start,
            PULocationID,
            COUNT(*) AS num_trips
        FROM TABLE(
            TUMBLE(TABLE green_trips, DESCRIPTOR(event_timestamp), INTERVAL '5' MINUTE)
        )
        GROUP BY window_start, PULocationID;
    """).wait()


if __name__ == '__main__':
    run()
```

Submit:
```bash
docker exec -it workshop-jobmanager-1 flink run -py /opt/src/job/q4_tumbling_pickup.py
```

Query:
```sql
SELECT PULocationID, num_trips
FROM tumbling_pickup
ORDER BY num_trips DESC
LIMIT 3;
```

**Answer: 74**

## Question 5: Session window - longest streak

Session window with 5-minute gap on `PULocationID`.

Save as `src/job/q5_session_pickup.py`:

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import EnvironmentSettings, StreamTableEnvironment


def create_source(t_env):
    t_env.execute_sql("""
        CREATE TABLE green_trips (
            lpep_pickup_datetime VARCHAR,
            lpep_dropoff_datetime VARCHAR,
            PULocationID INTEGER,
            DOLocationID INTEGER,
            passenger_count DOUBLE,
            trip_distance DOUBLE,
            tip_amount DOUBLE,
            total_amount DOUBLE,
            event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
            WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
        ) WITH (
            'connector' = 'kafka',
            'properties.bootstrap.servers' = 'redpanda:29092',
            'topic' = 'green-trips',
            'scan.startup.mode' = 'earliest-offset',
            'properties.auto.offset.reset' = 'earliest',
            'format' = 'json'
        );
    """)


def create_sink(t_env):
    t_env.execute_sql("""
        CREATE TABLE session_pickup (
            session_start TIMESTAMP(3),
            session_end TIMESTAMP(3),
            PULocationID INT,
            num_trips BIGINT
        ) WITH (
            'connector' = 'jdbc',
            'url' = 'jdbc:postgresql://postgres:5432/postgres',
            'table-name' = 'session_pickup',
            'username' = 'postgres',
            'password' = 'postgres',
            'driver' = 'org.postgresql.Driver'
        );
    """)


def run():
    env = StreamExecutionEnvironment.get_execution_environment()
    env.enable_checkpointing(10 * 1000)
    env.set_parallelism(1)

    settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
    t_env = StreamTableEnvironment.create(env, environment_settings=settings)

    create_source(t_env)
    create_sink(t_env)

    t_env.execute_sql("""
        INSERT INTO session_pickup
        SELECT
            session_start,
            session_end,
            PULocationID,
            COUNT(*) AS num_trips
        FROM TABLE(
            SESSION(TABLE green_trips PARTITION BY PULocationID,
                    DESCRIPTOR(event_timestamp), INTERVAL '5' MINUTE)
        )
        GROUP BY session_start, session_end, PULocationID;
    """).wait()


if __name__ == '__main__':
    run()
```

Submit:
```bash
docker exec -it workshop-jobmanager-1 flink run -py /opt/src/job/q5_session_pickup.py
```

Query:
```sql
SELECT PULocationID, num_trips, session_start, session_end
FROM session_pickup
ORDER BY num_trips DESC
LIMIT 3;
```

**Answer: 31**

## Question 6: Tumbling window - largest tip

1-hour tumbling window computing total `tip_amount` per hour.

Save as `src/job/q6_hourly_tips.py`:

```python
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.table import EnvironmentSettings, StreamTableEnvironment


def create_source(t_env):
    t_env.execute_sql("""
        CREATE TABLE green_trips (
            lpep_pickup_datetime VARCHAR,
            lpep_dropoff_datetime VARCHAR,
            PULocationID INTEGER,
            DOLocationID INTEGER,
            passenger_count DOUBLE,
            trip_distance DOUBLE,
            tip_amount DOUBLE,
            total_amount DOUBLE,
            event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
            WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
        ) WITH (
            'connector' = 'kafka',
            'properties.bootstrap.servers' = 'redpanda:29092',
            'topic' = 'green-trips',
            'scan.startup.mode' = 'earliest-offset',
            'properties.auto.offset.reset' = 'earliest',
            'format' = 'json'
        );
    """)


def create_sink(t_env):
    t_env.execute_sql("""
        CREATE TABLE hourly_tips (
            window_start TIMESTAMP(3),
            total_tips DOUBLE,
            PRIMARY KEY (window_start) NOT ENFORCED
        ) WITH (
            'connector' = 'jdbc',
            'url' = 'jdbc:postgresql://postgres:5432/postgres',
            'table-name' = 'hourly_tips',
            'username' = 'postgres',
            'password' = 'postgres',
            'driver' = 'org.postgresql.Driver'
        );
    """)


def run():
    env = StreamExecutionEnvironment.get_execution_environment()
    env.enable_checkpointing(10 * 1000)
    env.set_parallelism(1)

    settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
    t_env = StreamTableEnvironment.create(env, environment_settings=settings)

    create_source(t_env)
    create_sink(t_env)

    t_env.execute_sql("""
        INSERT INTO hourly_tips
        SELECT
            window_start,
            SUM(tip_amount) AS total_tips
        FROM TABLE(
            TUMBLE(TABLE green_trips, DESCRIPTOR(event_timestamp), INTERVAL '1' HOUR)
        )
        GROUP BY window_start;
    """).wait()


if __name__ == '__main__':
    run()
```

Submit:
```bash
docker exec -it workshop-jobmanager-1 flink run -py /opt/src/job/q6_hourly_tips.py
```

Query:
```sql
SELECT window_start, total_tips
FROM hourly_tips
ORDER BY total_tips DESC
LIMIT 3;
```

**Answer: 2025-10-01 18:00:00**